# SysMLDocGen MDK — Jupyter 演示

本 Notebook 演示 Jupyter MDK 的三大核心能力：
1. **API 客户端** — 连接系统、浏览项目/模型
2. **双向同步** — 拉取快照、本地编辑、对比差异、推回服务端
3. **DocGen 模板语言** — 编写模板、本地预览、推送服务端、生成文档

In [ ]:
# MDK 安装后会自动注册魔法命令，无需额外 import
# 如果魔法命令未生效，运行:
from sysmldocgen_mdk.jupyter import register_magics
register_magics()

## 1. 连接系统

使用 `%connect` 魔法命令一步完成连接与登录。

In [ ]:
%connect http://localhost:8000 admin admin123

## 2. 浏览项目和模型

使用魔法命令查看项目和模型列表。

In [ ]:
%list_projects

In [ ]:
%list_models
# 或按项目筛选: %list_models --project 1

## 3. 加载模型到本地

`%load_model` 拉取模型数据到本地变量，便于后续分析。

In [ ]:
# 将 model_id 改为你要分析的模型 ID
%load_model 2

### 分析模型数据

加载后自动创建了 `model_2_elements` 变量，直接用 pandas 分析。

In [ ]:
import pandas as pd
from sysmldocgen_mdk.jupyter import display_stats, display_table

df = model_2_elements
display_table(df)
display_stats(df)

In [ ]:
# 只看需求
reqs = df[df['element_type'] == 'Requirement']
reqs[['element_id', 'element_name', 'description']]

## 4. 双向同步 — 拉取快照

`%pull` 创建完整的同步快照，支持变更跟踪。

In [ ]:
%pull 2

### 本地编辑元素

通过快照对象的 `edit_element` 方法修改，自动记录变更。

In [ ]:
# 查看有哪些变更方法
snapshot = snapshot_2
print(f"快照包含 {len(snapshot.elements)} 个元素, {len(snapshot.relationships)} 条关系")
print(f"本地变更: {len(snapshot.local_changes)} 条")

# 按 element_id 查找并修改
first_elem = snapshot.elements.iloc[0]
print(f"第一个元素: {first_elem['element_id']} - {first_elem['element_name']}")

# 修改描述（这里只是演示，实际使用时取消注释并替换 element_id）
# snapshot.edit_element("req_1", description="已审核：新增需求描述")
# print(f"修改后本地变更: {len(snapshot.local_changes)} 条")

### 查看差异

如果有本地修改，可以对比服务端最新状态。

In [ ]:
# 对比本地快照与服务端差异
session = session_2
diff = session.diff(snapshot)
print(f"差异摘要: {diff.summary}")

### 推送变更

将本地修改推回服务端（请先取消上面 edit 单元格的注释）。

In [ ]:
# %sync_push 2
# 或带冲突跳过: %sync_push 2 --skip-conflicts

## 5. DocGen 模板语言 — 编写并预览

在单元格中编写 Jinja2 模板，使用 `%%render_doc` 实时预览。
支持 SysML 专用过滤器：`req_table`, `block_hierarchy`, `iface_matrix`, `format_attrs`。

In [ ]:
%%render_doc 需求规格说明书
# {{ model_name }} 需求规格说明书

**版本**: {{ model_version }}  
**生成时间**: {{ generate_time }}  
**元素总数**: {{ element_count }}

---

## 需求列表

{{ requirement_list | req_table("需求清单") }}

### Block 层级结构

{{ elements | block_hierarchy }}

## 6. 推送模板到服务端

将编写好的 DocGen 模板保存到服务端，供文档生成使用。

In [ ]:
from sysmldocgen_mdk import DocGenTemplate
from sysmldocgen_mdk import Client

tpl_content = """# {{ model_name }} 文档

## 需求
{{ requirement_list | req_table("需求清单") }}

## 系统结构
{{ elements | block_hierarchy }}
"""

tpl = DocGenTemplate(tpl_content, name="标准需求文档")
client = Client("http://localhost:8000", "admin", "admin123")
result = tpl.to_server(client, template_type="docgen")
print(f"已保存模板 ID: {result['id']}, 名称: {result['name']}")

## 7. 生成文档

用服务端模板生成文档。

In [ ]:
# 列出可用模板
templates = client.list_templates()
templates_df = pd.DataFrame(templates)
templates_df[['id', 'name', 'template_type', 'status']]

# 选择一个模板生成文档
# if templates:
#     doc = client.generate_document(
#         project_id=1,
#         model_id=2,
#         template_id=templates[0]["id"],
#     )
#     print(f"文档 ID: {doc['id']}, 状态: {doc['status']}")

## 8. 从服务端拉取已有模板

在 Jupyter 中编辑服务端已有的模板内容。

In [ ]:
# 从服务端加载模板
# if templates:
#     server_tpl = DocGenTemplate.from_server(client, templates[0]["id"])
#     print(f"加载模板: {server_tpl}")
#     print(server_tpl.content[:200])

---
## 更多资源

- **DocGen 语言教程**: 打开 `notebooks/docgen_tutorial.ipynb` 学习完整模板语法
- **API 参考**: `Client` 类封装了全部 REST API
- **同步引擎**: `SyncSession` 支持拉取、对比、推送、冲突检测